# Cell 1: Project Overview

Description: This notebook downloads a YouTube video with `yt-dlp` and extracts frames at a fixed time interval using OpenCV.

Run the cells from top to bottom.

In [2]:
# Cell 2: Install Dependencies
# Description: Installs the Python packages required to download videos and process frames.
%pip install opencv-python yt-dlp

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Cell 3: Import Libraries
# Description: Imports the modules used for filesystem work, video processing, and YouTube downloads.
import os
import cv2
import yt_dlp

In [5]:
# Cell 4: Define Video Download Function
# Description: Creates a helper function that downloads a YouTube video to a local MP4 file.
def download_video(url, output_path="video.mp4"):
    ydl_opts = {
        "outtmpl": output_path,
        "format": "mp4",
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return output_path

In [6]:
# Cell 5: Define Frame Extraction Function
# Description: Creates a helper function that saves one frame every few seconds from the downloaded video.
def extract_frames(video_path, interval_seconds=5, output_folder="frames"):
    os.makedirs(output_folder, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        cap.release()
        raise ValueError("Could not read FPS from the video file.")

    frame_interval = max(1, int(fps * interval_seconds))
    frame_index = 0
    saved_frames = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_index % frame_interval == 0:
            timestamp = frame_index / fps
            filename = f"{output_folder}/frame_{timestamp:.2f}s.jpg"
            cv2.imwrite(filename, frame)
            saved_frames.append({"timestamp": timestamp, "file": filename})

        frame_index += 1

    cap.release()
    print(f"Extracted {len(saved_frames)} frames.")
    return saved_frames

In [7]:
# Cell 6: Configure Inputs
# Description: Sets the video URL, output file name, frame interval, and output folder.
url = "https://youtu.be/55oPmIBq3Ok"
output_video = "video.mp4"
frame_interval_seconds = 5
output_folder = "frames"

In [8]:
# Cell 7: Run the Project
# Description: Downloads the video, extracts frames, and displays the saved frame list.
video_path = download_video(url, output_path=output_video)
frames = extract_frames(
    video_path,
    interval_seconds=frame_interval_seconds,
    output_folder=output_folder,
)
frames

[youtube] Extracting URL: https://youtu.be/55oPmIBq3Ok
[youtube] 55oPmIBq3Ok: Downloading webpage


[youtube] 55oPmIBq3Ok: Downloading android vr player API JSON
[info] 55oPmIBq3Ok: Downloading 1 format(s): 18
[download] video.mp4 has already been downloaded
[download] 100% of   15.76MiB
Extracted 80 frames.


[{'timestamp': 0.0, 'file': 'frames/frame_0.00s.jpg'},
 {'timestamp': 4.963291666666667, 'file': 'frames/frame_4.96s.jpg'},
 {'timestamp': 9.926583333333333, 'file': 'frames/frame_9.93s.jpg'},
 {'timestamp': 14.889874999999998, 'file': 'frames/frame_14.89s.jpg'},
 {'timestamp': 19.853166666666667, 'file': 'frames/frame_19.85s.jpg'},
 {'timestamp': 24.816458333333333, 'file': 'frames/frame_24.82s.jpg'},
 {'timestamp': 29.779749999999996, 'file': 'frames/frame_29.78s.jpg'},
 {'timestamp': 34.74304166666666, 'file': 'frames/frame_34.74s.jpg'},
 {'timestamp': 39.70633333333333, 'file': 'frames/frame_39.71s.jpg'},
 {'timestamp': 44.669624999999996, 'file': 'frames/frame_44.67s.jpg'},
 {'timestamp': 49.63291666666667, 'file': 'frames/frame_49.63s.jpg'},
 {'timestamp': 54.59620833333333, 'file': 'frames/frame_54.60s.jpg'},
 {'timestamp': 59.55949999999999, 'file': 'frames/frame_59.56s.jpg'},
 {'timestamp': 64.52279166666666, 'file': 'frames/frame_64.52s.jpg'},
 {'timestamp': 69.48608333333333

In [9]:
# Cell 8: Retrieval Setup (CLIP + FAISS)
# Description: Installs retrieval dependencies, imports libraries, and loads CLIP once.
%pip install torch torchvision pillow transformers faiss-cpu

import json
import numpy as np
import faiss
from PIL import Image
import torch
from transformers import CLIPModel, CLIPProcessor

if not frames:
    raise ValueError("No frames found. Run Cell 7 first to extract frames.")

model_name = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading CLIP processor...")
processor = CLIPProcessor.from_pretrained(model_name)
print("Loading CLIP model...")
model = CLIPModel.from_pretrained(model_name).to(device)
model.eval()

def to_embedding_tensor(output, embed_attr):
    if isinstance(output, torch.Tensor):
        return output
    if hasattr(output, embed_attr):
        return getattr(output, embed_attr)
    if hasattr(output, "pooler_output"):
        return output.pooler_output
    if isinstance(output, (tuple, list)) and len(output) > 0:
        return output[0]
    raise TypeError(f"Could not extract embedding tensor from output type: {type(output)}")

def l2_normalize(x):
    return x / x.norm(dim=-1, keepdim=True).clamp(min=1e-12)

print("Setup complete.")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Using device: cpu
Loading CLIP processor...


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading CLIP model...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 15563.86it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup complete.


## Phase A: Offline Index Build

Extract frames (already done in Cell 7), compute image embeddings once, and save FAISS index plus metadata to disk.

In [10]:
# Cell 9: Phase A (Offline) - Build and Save Embedding Index
# Description: Computes image embeddings once and stores FAISS index + metadata for fast reuse.

index_path = "clip.index"
metadata_path = "clip_metadata.json"
batch_size = 32

image_files = [item["file"] for item in frames if os.path.exists(item["file"])]
if not image_files:
    raise ValueError("No frame image files were found on disk.")

timestamp_by_file = {item["file"]: item["timestamp"] for item in frames}
records = [
    {"file": path, "timestamp": timestamp_by_file.get(path)}
    for path in image_files
]

print(f"Encoding {len(image_files)} frame images in batches of {batch_size}...")
image_chunks = []

for start in range(0, len(image_files), batch_size):
    end = min(start + batch_size, len(image_files))
    batch_paths = image_files[start:end]
    batch_images = []
    for path in batch_paths:
        with Image.open(path) as img:
            batch_images.append(img.convert("RGB"))

    image_inputs = processor(images=batch_images, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        image_output = model.get_image_features(pixel_values=image_inputs["pixel_values"])

    image_embeds = to_embedding_tensor(image_output, "image_embeds")
    if not isinstance(image_embeds, torch.Tensor):
        raise TypeError(f"Expected tensor embeddings, got {type(image_embeds)}")

    image_embeds = l2_normalize(image_embeds)
    image_chunks.append(image_embeds.detach().cpu())
    print(f"Processed {end}/{len(image_files)} images")

image_matrix = torch.cat(image_chunks, dim=0).numpy().astype("float32")

dim = image_matrix.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(image_matrix)

faiss.write_index(index, index_path)
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump({"model_name": model_name, "records": records}, f, indent=2)

print(f"Saved FAISS index to {index_path}")
print(f"Saved metadata to {metadata_path}")
print(f"Indexed vectors: {index.ntotal}")

Encoding 80 frame images in batches of 32...
Processed 32/80 images
Processed 64/80 images
Processed 80/80 images
Saved FAISS index to clip.index
Saved metadata to clip_metadata.json
Indexed vectors: 80


## Phase B: Online Query

Load the saved embeddings/index, embed only the text query, and search the nearest frames.

In [12]:
# Cell 10: Phase B (Online) - Query the Saved Index
# Description: Loads FAISS index/metadata, embeds query text, and returns top matching frames.

index_path = "clip.index"
metadata_path = "clip_metadata.json"
query = "3 scenes"

if not os.path.exists(index_path) or not os.path.exists(metadata_path):
    raise FileNotFoundError("Run Cell 9 first to build and save the index.")

print("Loading FAISS index and metadata...")
index = faiss.read_index(index_path)
with open(metadata_path, "r", encoding="utf-8") as f:
    meta = json.load(f)
records = meta["records"]

print(f"Embedding query: {query}")
text_inputs = processor(text=[query], return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    text_output = model.get_text_features(
        input_ids=text_inputs["input_ids"],
        attention_mask=text_inputs["attention_mask"],
    )

text_embeds = to_embedding_tensor(text_output, "text_embeds")
text_embeds = l2_normalize(text_embeds)
query_vec = text_embeds.detach().cpu().numpy().astype("float32")

top_k = min(5, index.ntotal)
scores, indices = index.search(query_vec, top_k)

retrieved = []
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    item = records[int(idx)]
    retrieved.append(
        {
            "rank": rank,
            "score": float(score),
            "timestamp": item.get("timestamp"),
            "file": item.get("file"),
        }
    )

retrieved

Loading FAISS index and metadata...
Embedding query: 3 scenes


[{'rank': 1,
  'score': 0.2604251801967621,
  'timestamp': 74.44937499999999,
  'file': 'frames/frame_74.45s.jpg'},
 {'rank': 2,
  'score': 0.2600771188735962,
  'timestamp': 124.08229166666666,
  'file': 'frames/frame_124.08s.jpg'},
 {'rank': 3,
  'score': 0.2579365372657776,
  'timestamp': 109.19241666666666,
  'file': 'frames/frame_109.19s.jpg'},
 {'rank': 4,
  'score': 0.2563464045524597,
  'timestamp': 188.6050833333333,
  'file': 'frames/frame_188.61s.jpg'},
 {'rank': 5,
  'score': 0.2497999370098114,
  'timestamp': 342.46712499999995,
  'file': 'frames/frame_342.47s.jpg'}]

## Sound Search Method (Text -> Audio Moments)

Plan:

1. Setup a CLAP audio-text model and audio utilities.
2. Phase A (Offline): extract audio, split into time windows, compute audio embeddings, save FAISS index + metadata.
3. Phase B (Online): embed text query, search FAISS, return matching time ranges in the video.

In [13]:
# Cell 11: Sound Search Setup (CLAP + FAISS)
# Description: Installs audio retrieval dependencies and loads CLAP once.
%pip install librosa soundfile imageio-ffmpeg

import subprocess
import librosa
import imageio_ffmpeg
from transformers import AutoProcessor, ClapModel

audio_model_name = "laion/clap-htsat-unfused"
audio_device = device if "device" in globals() else ("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device for audio retrieval: {audio_device}")
print("Loading CLAP processor...")
audio_processor = AutoProcessor.from_pretrained(audio_model_name)
print("Loading CLAP model...")
audio_model = ClapModel.from_pretrained(audio_model_name).to(audio_device)
audio_model.eval()

def format_seconds(total_seconds):
    total_seconds = max(0.0, float(total_seconds))
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = total_seconds % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:05.2f}"

print("Sound search setup complete.")

Note: you may need to restart the kernel to use updated packages.
Using device for audio retrieval: cpu
Loading CLAP processor...



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Loading CLAP model...


Loading weights: 100%|██████████| 447/447 [00:00<00:00, 13718.21it/s]

Sound search setup complete.


## Sound Search Phase A: Offline Index Build

Extract audio from the video, chunk it into windows, encode windows with CLAP, and save FAISS + metadata.

In [14]:
# Cell 12: Sound Search Phase A (Offline) - Build Audio Index
# Description: Builds an audio FAISS index from overlapping audio windows.

audio_index_path = "audio_clap.index"
audio_metadata_path = "audio_clap_metadata.json"
audio_wav_path = "video_audio.wav"
window_seconds = 2.0
hop_seconds = 1.0
audio_batch_size = 16

if not os.path.exists(output_video):
    raise FileNotFoundError("Run Cell 7 first to create the video file.")

target_sr = int(audio_processor.feature_extractor.sampling_rate)
ffmpeg_bin = imageio_ffmpeg.get_ffmpeg_exe()
print("Extracting audio track from video...")
subprocess.run(
    [
        ffmpeg_bin,
        "-y",
        "-i", output_video,
        "-vn",
        "-ac", "1",
        "-ar", str(target_sr),
        audio_wav_path,
    ],
    check=True,
    capture_output=True,
    text=True,
 )

audio_signal, sr = librosa.load(audio_wav_path, sr=target_sr, mono=True)
if audio_signal.size == 0:
    raise ValueError("Extracted audio is empty.")

window_samples = max(1, int(window_seconds * sr))
hop_samples = max(1, int(hop_seconds * sr))
if len(audio_signal) < window_samples:
    audio_signal = np.pad(audio_signal, (0, window_samples - len(audio_signal)))

windows = []
audio_records = []
for start in range(0, len(audio_signal) - window_samples + 1, hop_samples):
    end = start + window_samples
    start_sec = start / sr
    end_sec = end / sr
    windows.append(audio_signal[start:end])
    audio_records.append({
        "start": float(start_sec),
        "end": float(end_sec),
    })

if not windows:
    raise ValueError("No audio windows were generated.")

print(f"Encoding {len(windows)} audio windows in batches of {audio_batch_size}...")
audio_chunks = []
for start in range(0, len(windows), audio_batch_size):
    end = min(start + audio_batch_size, len(windows))
    batch_audio = windows[start:end]
    audio_inputs = audio_processor(
        audio=batch_audio,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True,
    ).to(audio_device)

    with torch.no_grad():
        audio_output = audio_model.get_audio_features(
            input_features=audio_inputs["input_features"],
            is_longer=audio_inputs.get("is_longer"),
        )

    audio_embeds = to_embedding_tensor(audio_output, "audio_embeds")
    if not isinstance(audio_embeds, torch.Tensor):
        raise TypeError(f"Expected tensor audio embeddings, got {type(audio_embeds)}")

    audio_embeds = l2_normalize(audio_embeds)
    audio_chunks.append(audio_embeds.detach().cpu())
    print(f"Processed {end}/{len(windows)} audio windows")

audio_matrix = torch.cat(audio_chunks, dim=0).numpy().astype("float32")

audio_dim = audio_matrix.shape[1]
audio_index = faiss.IndexFlatIP(audio_dim)
audio_index.add(audio_matrix)

faiss.write_index(audio_index, audio_index_path)
with open(audio_metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "audio_model_name": audio_model_name,
            "window_seconds": window_seconds,
            "hop_seconds": hop_seconds,
            "sampling_rate": sr,
            "records": audio_records,
        },
        f,
        indent=2,
    )

print(f"Saved audio FAISS index to {audio_index_path}")
print(f"Saved audio metadata to {audio_metadata_path}")
print(f"Indexed audio windows: {audio_index.ntotal}")

Extracting audio track from video...
Encoding 394 audio windows in batches of 16...
Processed 16/394 audio windows
Processed 32/394 audio windows
Processed 48/394 audio windows
Processed 64/394 audio windows
Processed 80/394 audio windows
Processed 96/394 audio windows
Processed 112/394 audio windows
Processed 128/394 audio windows
Processed 144/394 audio windows
Processed 160/394 audio windows
Processed 176/394 audio windows
Processed 192/394 audio windows
Processed 208/394 audio windows
Processed 224/394 audio windows
Processed 240/394 audio windows
Processed 256/394 audio windows
Processed 272/394 audio windows
Processed 288/394 audio windows
Processed 304/394 audio windows
Processed 320/394 audio windows
Processed 336/394 audio windows
Processed 352/394 audio windows
Processed 368/394 audio windows
Processed 384/394 audio windows
Processed 394/394 audio windows
Saved audio FAISS index to audio_clap.index
Saved audio metadata to audio_clap_metadata.json
Indexed audio windows: 394


## Sound Search Phase B: Online Query

Embed only the query text and retrieve matching audio time ranges from the saved index.

In [19]:
# Cell 13: Sound Search Phase B (Online) - Text Query to Audio Moments
# Description: Loads audio index, embeds text query, and returns top matching time windows with MM:SS display.

audio_index_path = "audio_clap.index"
audio_metadata_path = "audio_clap_metadata.json"
audio_query = "guitar"
audio_top_k = 5

if not os.path.exists(audio_index_path) or not os.path.exists(audio_metadata_path):
    raise FileNotFoundError("Run Cell 12 first to build and save the audio index.")

def to_mmss(seconds):
    total = max(0.0, float(seconds))
    minutes = int(total // 60)
    secs = int(round(total % 60))
    if secs == 60:
        minutes += 1
        secs = 0
    return f"{minutes:02d}:{secs:02d}"

print("Loading audio FAISS index and metadata...")
audio_index = faiss.read_index(audio_index_path)
with open(audio_metadata_path, "r", encoding="utf-8") as f:
    audio_meta = json.load(f)
audio_records = audio_meta["records"]

print(f"Embedding audio query: {audio_query}")
text_inputs = audio_processor(text=[audio_query], return_tensors="pt", padding=True).to(audio_device)
with torch.no_grad():
    text_output = audio_model.get_text_features(
        input_ids=text_inputs["input_ids"],
        attention_mask=text_inputs["attention_mask"],
    )

audio_query_embed = to_embedding_tensor(text_output, "text_embeds")
if not isinstance(audio_query_embed, torch.Tensor):
    raise TypeError(f"Expected tensor text embeddings, got {type(audio_query_embed)}")

audio_query_embed = l2_normalize(audio_query_embed)
audio_query_vec = audio_query_embed.detach().cpu().numpy().astype("float32")

k = min(audio_top_k, audio_index.ntotal)
audio_scores, audio_indices = audio_index.search(audio_query_vec, k)

audio_retrieved = []
for rank, (idx, score) in enumerate(zip(audio_indices[0], audio_scores[0]), start=1):
    item = audio_records[int(idx)]
    start_sec = float(item["start"])
    end_sec = float(item["end"])
    start_minute = start_sec / 60.0
    end_minute = end_sec / 60.0
    audio_retrieved.append(
        {
            "rank": rank,
            "score": float(score),
            "start_mmss": to_mmss(start_sec),
            "end_mmss": to_mmss(end_sec),
            "time_range": f"{to_mmss(start_sec)} -> {to_mmss(end_sec)}",
            "start_minute_decimal": round(start_minute, 2),
            "end_minute_decimal": round(end_minute, 2),
        }
    )

audio_retrieved

Loading audio FAISS index and metadata...
Embedding audio query: guitar


[{'rank': 1,
  'score': 0.5312962532043457,
  'start_mmss': '05:48',
  'end_mmss': '05:50',
  'time_range': '05:48 -> 05:50',
  'start_minute_decimal': 5.8,
  'end_minute_decimal': 5.83},
 {'rank': 2,
  'score': 0.5094094276428223,
  'start_mmss': '01:55',
  'end_mmss': '01:57',
  'time_range': '01:55 -> 01:57',
  'start_minute_decimal': 1.92,
  'end_minute_decimal': 1.95},
 {'rank': 3,
  'score': 0.4946895241737366,
  'start_mmss': '05:36',
  'end_mmss': '05:38',
  'time_range': '05:36 -> 05:38',
  'start_minute_decimal': 5.6,
  'end_minute_decimal': 5.63},
 {'rank': 4,
  'score': 0.4941711127758026,
  'start_mmss': '05:49',
  'end_mmss': '05:51',
  'time_range': '05:49 -> 05:51',
  'start_minute_decimal': 5.82,
  'end_minute_decimal': 5.85},
 {'rank': 5,
  'score': 0.4892573952674866,
  'start_mmss': '05:40',
  'end_mmss': '05:42',
  'time_range': '05:40 -> 05:42',
  'start_minute_decimal': 5.67,
  'end_minute_decimal': 5.7}]

## Speech Search Method (Whisper: Speech -> Text Moments)

Plan:

1. Setup Whisper speech-to-text model.
2. Phase A (Offline): extract audio, split into windows, transcribe each window, save metadata.
3. Phase B (Online): match text query against transcriptions and return matching time ranges.

In [ ]:
# Cell 14: Whisper Setup (Speech-to-Text)
# Description: Loads Whisper model/processor for speech transcription.

%pip install sentencepiece

from transformers import WhisperProcessor, WhisperForConditionalGeneration

speech_model_name = "openai/whisper-small"
speech_device = device if "device" in globals() else ("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device for speech transcription: {speech_device}")
print("Loading Whisper processor...")
speech_processor = WhisperProcessor.from_pretrained(speech_model_name)
print("Loading Whisper model...")
speech_model = WhisperForConditionalGeneration.from_pretrained(speech_model_name).to(speech_device)
speech_model.eval()

# Configure language/task once to avoid passing them repeatedly in generate().
speech_model.generation_config.language = "en"
speech_model.generation_config.task = "transcribe"
speech_model.generation_config.forced_decoder_ids = None

print("Whisper setup complete.")

In [ ]:
# Cell 15: Speech Search Phase A (Offline) - Build Whisper Transcription Index
# Description: Extracts audio, creates overlapping windows, transcribes each window, and saves metadata.

speech_metadata_path = "speech_whisper_metadata.json"
speech_audio_wav_path = "video_audio_whisper.wav"
speech_window_seconds = 2.0
speech_hop_seconds = 1.0

if not os.path.exists(output_video):
    raise FileNotFoundError("Run Cell 7 first to create the video file.")

speech_target_sr = 16000
ffmpeg_bin = imageio_ffmpeg.get_ffmpeg_exe()
print("Extracting audio track for Whisper...")
subprocess.run(
    [
        ffmpeg_bin,
        "-y",
        "-i", output_video,
        "-vn",
        "-ac", "1",
        "-ar", str(speech_target_sr),
        speech_audio_wav_path,
    ],
    check=True,
    capture_output=True,
    text=True,
)

speech_signal, speech_sr = librosa.load(speech_audio_wav_path, sr=speech_target_sr, mono=True)
if speech_signal.size == 0:
    raise ValueError("Extracted speech audio is empty.")

speech_window_samples = max(1, int(speech_window_seconds * speech_sr))
speech_hop_samples = max(1, int(speech_hop_seconds * speech_sr))
if len(speech_signal) < speech_window_samples:
    speech_signal = np.pad(speech_signal, (0, speech_window_samples - len(speech_signal)))

speech_windows = []
speech_records = []
for start in range(0, len(speech_signal) - speech_window_samples + 1, speech_hop_samples):
    end = start + speech_window_samples
    start_sec = start / speech_sr
    end_sec = end / speech_sr
    speech_windows.append(speech_signal[start:end])
    speech_records.append({
        "start": float(start_sec),
        "end": float(end_sec),
    })

if not speech_windows:
    raise ValueError("No speech windows were generated.")

print(f"Transcribing {len(speech_windows)} audio windows with Whisper...")
transcription_records = []
for i, w in enumerate(speech_windows):
    whisper_inputs = speech_processor(
        w,
        sampling_rate=speech_sr,
        return_tensors="pt",
        return_attention_mask=True,
    ).to(speech_device)

    with torch.no_grad():
        generated_ids = speech_model.generate(
            input_features=whisper_inputs["input_features"],
            attention_mask=whisper_inputs.get("attention_mask"),
        )

    text = speech_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    transcription_records.append({
        "start": speech_records[i]["start"],
        "end": speech_records[i]["end"],
        "text": text,
    })

    if (i + 1) % max(1, len(speech_windows) // 10) == 0 or (i + 1) == len(speech_windows):
        print(f"[transcribe] {i + 1}/{len(speech_windows)} windows done")

with open(speech_metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "speech_model_name": speech_model_name,
            "sampling_rate": speech_sr,
            "window_seconds": speech_window_seconds,
            "hop_seconds": speech_hop_seconds,
            "records": transcription_records,
        },
        f,
        indent=2,
    )

print(f"Saved speech metadata to {speech_metadata_path}")
print(f"Transcribed windows: {len(transcription_records)}")

In [ ]:
# Cell 16: Speech Search Phase B (Online) - Text Query to Spoken Moments
# Description: Loads Whisper metadata, matches text query, and returns matching time windows with MM:SS display.

speech_metadata_path = "speech_whisper_metadata.json"
speech_query = "hello"
speech_top_k = 5

if not os.path.exists(speech_metadata_path):
    raise FileNotFoundError("Run Cell 15 first to build and save speech metadata.")

with open(speech_metadata_path, "r", encoding="utf-8") as f:
    speech_meta = json.load(f)
speech_records = speech_meta["records"]

def to_mmss(seconds):
    total = max(0.0, float(seconds))
    minutes = int(total // 60)
    secs = int(round(total % 60))
    if secs == 60:
        minutes += 1
        secs = 0
    return f"{minutes:02d}:{secs:02d}"

query_lower = speech_query.lower().strip()
if not query_lower:
    raise ValueError("speech_query must not be empty")

matches = []
for item in speech_records:
    text = (item.get("text") or "").strip()
    text_lower = text.lower()
    if query_lower in text_lower:
        count = text_lower.count(query_lower)
        pos = text_lower.find(query_lower)
        score = float(count) / (1 + pos / max(1, len(text_lower)))
        matches.append((score, item))

matches.sort(key=lambda x: x[0], reverse=True)

speech_retrieved = []
for rank, (score, item) in enumerate(matches[:speech_top_k], start=1):
    start_sec = float(item["start"])
    end_sec = float(item["end"])
    speech_retrieved.append(
        {
            "rank": rank,
            "score": float(score),
            "start_mmss": to_mmss(start_sec),
            "end_mmss": to_mmss(end_sec),
            "time_range": f"{to_mmss(start_sec)} -> {to_mmss(end_sec)}",
            "text": item.get("text", ""),
        }
    )

speech_retrieved